# 📉 Fallstudie 1: Customer Churn

**Szenario**: Du bist Data Analyst bei TelcoMax GmbH. Auftrag: Analysiere, welche Kunden das Unternehmen verlassen ("Churn"), und bereite die Daten für ein Vorhersagemodell vor.

**Vollständiger DAV-Workflow:**
1. Business Understanding
2. Data Understanding (EDA)
3. Data Preparation (Bereinigen + Transformieren)
4. Vorbereitung für ML
5. Insights & Empfehlungen

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import LabelEncoder, StandardScaler
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('=== Fallstudie: Customer Churn ===')
print('Unternehmen: TelcoMax GmbH')

## Phase 1: Business Understanding

**Geschäftsfrage**: Welche Kunden werden das Unternehmen verlassen?

**Warum wichtig?**
- Neukunden zu gewinnen kostet 5-7x mehr als einen Bestandskunden zu halten
- 1% weniger Churn = signifikante Umsatzverbesserung

**KPIs:**
- Churn-Rate (Anteil abwandernder Kunden)
- Wichtigste Churn-Faktoren (Feature Importance)
- Hochrisiko-Kundensegmente

In [ ]:
# Phase 2: Data Understanding
df = pd.read_csv('../data/customer_churn.csv')
print(f'Datensatz: {df.shape[0]:,} Kunden, {df.shape[1]} Merkmale')
print(f'Churn-Rate: {df["churn"].mean()*100:.1f}%')
print(f'Absolute Zahlen: {df["churn"].sum()} Churner von {len(df)}')
print()
print(df.head(5))
print()
print('Fehlende Werte:')
print(df.isnull().sum()[df.isnull().sum()>0])

In [ ]:
# EDA: Churn nach verschiedenen Dimensionen
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Churn-Rate nach Vertragstyp', 'Churn-Rate nach Internetservice',
                    'Monatl. Kosten: Churn vs. Kein Churn', 'Laufzeit: Churn vs. Kein Churn'],
    vertical_spacing=0.15)

# Churn nach Contract Type
ct = df.groupby('contract_type')['churn'].mean().reset_index()
fig.add_trace(go.Bar(x=ct['contract_type'], y=ct['churn']*100,
    marker_color=['#0389CF','#5BB8F5','#E8792F'], text=[f'{v:.1f}%' for v in ct['churn']*100],
    textposition='outside', name='Churn-Rate'), row=1, col=1)

# Churn nach Internet Service
is_churn = df.groupby('internet_service')['churn'].mean().reset_index()
fig.add_trace(go.Bar(x=is_churn['internet_service'], y=is_churn['churn']*100,
    marker_color='#0389CF', text=[f'{v:.1f}%' for v in is_churn['churn']*100],
    textposition='outside', name='Churn nach Service', showlegend=False), row=1, col=2)

# Box: Monthly Charge
for churn_val, color, name in [(0,'#0389CF','Kein Churn'), (1,'#E8792F','Churn')]:
    subset = df[df['churn']==churn_val]['monthly_charge']
    fig.add_trace(go.Box(y=subset, name=name, marker_color=color, showlegend=True if churn_val==0 else False), row=2, col=1)

# Box: Tenure
for churn_val, color in [(0,'#0389CF'), (1,'#E8792F')]:
    subset = df[df['churn']==churn_val]['tenure_months']
    fig.add_trace(go.Box(y=subset, name=['Kein Churn','Churn'][churn_val],
                         marker_color=color, showlegend=False), row=2, col=2)

fig.update_layout(height=600, title_text='Customer Churn – Explorative Analyse',
                  title_font_size=14)
fig.show()

In [ ]:
# Phase 3: Data Preparation
print('=== Data Preparation ===')

df_prep = df.copy()

# 1. Fehlende Werte
df_prep['total_charges'] = pd.to_numeric(df_prep['total_charges'], errors='coerce')
df_prep['total_charges'] = df_prep['total_charges'].fillna(df_prep['total_charges'].median())
print(f'Fehlende Werte: {df_prep.isnull().sum().sum()}')

# 2. Encoding
le = LabelEncoder()
cat_cols = ['contract_type', 'internet_service', 'tech_support']
for col in cat_cols:
    df_prep[col+'_enc'] = le.fit_transform(df_prep[col])
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# 3. Feature Engineering
df_prep['total_charges_per_month'] = (df_prep['total_charges'] / df_prep['tenure_months'].replace(0,1)).round(2)
df_prep['is_longterm'] = (df_prep['tenure_months'] > 24).astype(int)
df_prep['high_charges'] = (df_prep['monthly_charge'] > df_prep['monthly_charge'].quantile(0.75)).astype(int)

# 4. Scaling
num_cols = ['age','monthly_charge','total_charges','tenure_months','total_charges_per_month']
scaler = StandardScaler()
df_prep[num_cols] = scaler.fit_transform(df_prep[num_cols])

print(f'\nML-bereiter Datensatz: {df_prep.shape}')

In [ ]:
# Feature Importance (vereinfacht: Korrelation mit Churn)
enc_cols = [c for c in df_prep.columns if c.endswith('_enc') or c in ['monthly_charge','tenure_months','is_longterm','high_charges']]
corr_churn = df_prep[enc_cols + ['churn']].corr()['churn'].drop('churn').abs().sort_values(ascending=False)

print('=== Feature-Korrelation mit Churn (abs.) ===')
print(corr_churn.round(4).to_string())

fig = px.bar(corr_churn.reset_index(),
    x='index', y='churn',
    title='Feature-Wichtigkeit für Churn (Korrelation)',
    labels={'index':'Feature', 'churn':'|Korrelation|'},
    color='churn', color_continuous_scale='Blues')
fig.show()

In [ ]:
# In PostgreSQL laden
from sqlalchemy import create_engine
pg_host = os.environ.get('POSTGRES_HOST', 'postgres')
try:
    engine = create_engine(f'postgresql://dav:dav2026@{pg_host}:5432/davdb')
    df.to_sql('customer_churn_raw', engine, if_exists='replace', index=False)
    df_prep.to_sql('customer_churn_prepared', engine, if_exists='replace', index=False)
    print('✅ Daten in PostgreSQL geladen!')
    
    # SQL-Analyse
    query = """
        SELECT contract_type, 
               COUNT(*) as total,
               SUM(churn) as churner,
               ROUND(100.0 * SUM(churn) / COUNT(*), 1) as churn_rate
        FROM customer_churn_raw
        GROUP BY contract_type
        ORDER BY churn_rate DESC
    """
    df_sql = pd.read_sql(query, engine)
    print('\nSQL-Analyse Churn nach Vertragstyp:')
    print(df_sql)
except Exception as e:
    print(f'PostgreSQL: {e}')

## Insights & Empfehlungen

**Erkenntnisse:**
1. **Month-to-month Verträge** haben die höchste Churn-Rate – Anreize für längere Laufzeiten setzen
2. **Fiber Optic-Kunden** churnen häufiger – Servicequalität prüfen
3. **Kurze Laufzeit** ist stärkster Churn-Prädiktor – Onboarding verbessern
4. **Hohe monatliche Kosten** korrelieren mit Churn – Preismodell überdenken

**Nächste Schritte:** Logistische Regression oder Random Forest auf den vorbereiteten Daten trainieren.

---
**Weiter mit:** `10_Fallstudie_Movies.ipynb`